In [1]:
import pandas as pd

**Obtención de servicios DENUE por AGEB**

*Servicios DENUE 2020* 
Clave - Descripción de categoría
* 621610 - Servicios de enfermería a domicilio
* 623311 - Asilos y otras residencias del sector privado para el cuidado de ancianos
* 623312 - Asilos y otras residencias del sector público para el cuidado de ancianos

In [2]:
denue_2018 = pd.read_csv("../../data/raw/2018/denue2018.csv", encoding="latin-1")
denue_2018.info()

/tmp/ipykernel_52742/2025268452.py:1: DtypeWarning: Columns (14,34) have mixed types. Specify dtype option on import or set low_memory=False.
  denue_2018 = pd.read_csv("../../data/raw/2018/denue2018.csv", encoding="latin-1")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471957 entries, 0 to 471956
Data columns (total 41 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          471957 non-null  int64  
 1   nom_estab   471957 non-null  object 
 2   raz_social  99886 non-null   object 
 3   codigo_act  471957 non-null  int64  
 4   nombre_act  471957 non-null  object 
 5   per_ocu     471957 non-null  object 
 6   tipo_vial   469847 non-null  object 
 7   nom_vial    471456 non-null  object 
 8   tipo_v_e_1  457844 non-null  object 
 9   nom_v_e_1   466442 non-null  object 
 10  tipo_v_e_2  457748 non-null  object 
 11  nom_v_e_2   466371 non-null  object 
 12  tipo_v_e_3  456459 non-null  object 
 13  nom_v_e_3   466293 non-null  object 
 14  numero_ext  469068 non-null  object 
 15  letra_ext   190196 non-null  object 
 16  edificio    22480 non-null   object 
 17  edificio_e  36683 non-null   object 
 18  numero_int  81548 non-null   object 
 19  le

In [3]:
denue_2018.sample()

,id,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
285783,692522,DESPACHO CONTABLE,NaN,541211,Servicios de contabilidad y auditorÃ­a,0 a 5 personas,CALLE,SUR 128,AVENIDA,CONSTITUYENTES,...,Miguel Hidalgo ...,1103,15.0,5555150232,NaN,NaN,Fijo,19.405955,-99.202344,2014-12


Lo que unicamente utilizaremos sera:
* Código de la clase de actividad
* Entidad federativa
* Municipio 
* Localidad
* Área geoestadística básica

In [4]:
denue_2018 = denue_2018[["codigo_act", "cve_ent", "entidad", "cve_mun", "municipio", "cve_loc", "localidad", "ageb"]]
denue_2018.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471957 entries, 0 to 471956
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   codigo_act  471957 non-null  int64 
 1   cve_ent     471957 non-null  int64 
 2   entidad     471957 non-null  object
 3   cve_mun     471957 non-null  int64 
 4   municipio   471957 non-null  object
 5   cve_loc     471957 non-null  int64 
 6   localidad   471957 non-null  object
 7   ageb        471956 non-null  object
dtypes: int64(4), object(4)
memory usage: 28.8+ MB


Hay localidades no identificadas, pero esperemos a ver si dentro de nuestras actividades siguen habiendo.

In [5]:
denue_2018["cve_ent"] = denue_2018["cve_ent"].astype(str).str.zfill(2)
denue_2018["cve_mun"] = denue_2018["cve_mun"].astype(str).str.zfill(3)
denue_2018["cve_loc"] = denue_2018["cve_loc"].astype(str).str.zfill(4)
denue_2018["ageb"] = denue_2018["ageb"].str.zfill(4)
denue_2018.sample()

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb
111037,461160,09,CIUDAD DE MÃXICO,017,Venustiano Carranza,0001,Venustiano Carranza ...,0687


In [6]:
denue_2018["cvegeo"] = (denue_2018["cve_ent"] + denue_2018["cve_mun"] + denue_2018["cve_loc"] + denue_2018["ageb"])
denue_2018.sample()

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,cvegeo
48507,434224,09,CIUDAD DE MÃXICO,006,Iztacalco,0001,Iztacalco ...,1060,0900600011060


In [7]:
serv_enf_dom = denue_2018[denue_2018["codigo_act"] == 621610]

In [8]:
serv_enf_dom = serv_enf_dom.groupby("cvegeo", as_index=False)["codigo_act"].count()
serv_enf_dom.rename(columns={"codigo_act":"cantidad_serv_enf_dom"}, inplace=True)
serv_enf_dom

,cvegeo,cantidad_serv_enf_dom
0,0900200010839,1
1,0900500011415,1
2,0900500011557,1
3,090050001224A,1
4,0900600010382,1
5,0900700012174,1
6,0900700013825,2
7,090100001014A,1
8,0901000011260,2
9,0901200010695,1


In [9]:
asilos = denue_2018[denue_2018["codigo_act"].isin([623311, 623312])]
asilos

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,cvegeo
319936,623312,09,CIUDAD DE MÃXICO,014,Benito JuÃ¡rez,0001,Benito JuÃ¡rez ...,0384,0901400010384
319949,623312,09,CIUDAD DE MÃXICO,014,Benito JuÃ¡rez,0001,Benito JuÃ¡rez ...,0204,0901400010204
319950,623311,09,CIUDAD DE MÃXICO,016,Miguel Hidalgo,0001,Miguel Hidalgo ...,1156,0901600011156
320560,623311,09,CIUDAD DE MÃXICO,005,Gustavo A. Madero,0001,Gustavo A. Madero ...,1754,0900500011754
320561,623311,09,CIUDAD DE MÃXICO,008,La Magdalena Contreras,0001,La Magdalena Contreras ...,0283,0900800010283
...,...,...,...,...,...,...,...,...,...
341243,623311,09,CIUDAD DE MÃXICO,016,Miguel Hidalgo,0001,Miguel Hidalgo ...,1029,0901600011029
341244,623311,09,CIUDAD DE MÃXICO,012,Tlalpan,0001,Tlalpan ...,0958,0901200010958
341245,623311,09,CIUDAD DE MÃXICO,002,Azcapotzalco,0001,Azcapotzalco ...,0345,0900200010345
341246,623311,09,CIUDAD DE MÃXICO,014,Benito JuÃ¡rez,0001,Benito JuÃ¡rez ...,005A,090140001005A


In [10]:
asilos = asilos.groupby("cvegeo", as_index=False)["codigo_act"].count()
asilos.rename(columns={"codigo_act":"cantidad_asilos"}, inplace=True)
asilos

,cvegeo,cantidad_asilos
0,0900200010345,2
1,0900200010491,1
2,0900200010699,1
3,0900200010792,1
4,0900300010041,1
...,...,...
70,0901600010942,1
71,0901600011029,1
72,0901600011156,1
73,0901600011175,1


In [12]:
m = serv_enf_dom.merge(asilos, on="cvegeo", how="outer")

In [13]:
m.fillna(0)

,cvegeo,cantidad_serv_enf_dom,cantidad_asilos
0,0900200010345,0.0,2.0
1,0900200010491,0.0,1.0
2,0900200010699,0.0,1.0
3,0900200010792,0.0,1.0
4,0900200010839,1.0,0.0
...,...,...,...
93,0901600011156,0.0,1.0
94,0901600011175,0.0,1.0
95,0901700010634,1.0,0.0
96,0901700010920,0.0,1.0


In [14]:
m.to_csv("../../data/processed/datos_denue_2018.csv")